In [ ]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


In [7]:
X=torch.randn(100,1)*10
Y= 2*X+0.7 +torch.randn(100,1)

w = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, 1, requires_grad=True)

learning_rate = 0.001
epochs = 100
print(f"Initial Params -> w: {w.item():.3f}, b: {b.item():.3f}")

Initial Params -> w: -0.314, b: -0.356


In [8]:
for epoch in range(epochs):
    
    # --- A. Forward Pass ---
    # In TF Keras, this is implicit. Here, it's explicit math.
    preds = X @ w + b
    
    # --- B. Loss Calculation ---
    # Manual MSE: sum((y_pred - y_true)^2) / N
    loss = (preds - Y).pow(2).mean()
    
    # --- C. Backward Pass ---
    # This computes dLoss/dw and dLoss/db and stores them in w.grad and b.grad
    loss.backward()
    
    # --- D. Weight Update (The Critical Part) ---
    # We must wrap this in no_grad() because we don't want to track 
    # the update step itself in the computational graph.
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        
        # --- E. Zero Gradients ---
        # If we don't do this, gradients accumulate (add up) every epoch!
        w.grad.zero_()
        b.grad.zero_()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}, w = {w.item():.3f}, b = {b.item():.3f}")

print(f"\nFinal Params -> w: {w.item():.3f}, b: {b.item():.3f}")
print(f"Target Params -> w: 2.000, b: 0.700")

Epoch 10: Loss = 20.8981, w = 1.598, b = -0.324
Epoch 20: Loss = 2.3272, w = 1.938, b = -0.303
Epoch 30: Loss = 1.7086, w = 1.998, b = -0.285
Epoch 40: Loss = 1.6583, w = 2.009, b = -0.267
Epoch 50: Loss = 1.6272, w = 2.010, b = -0.249
Epoch 60: Loss = 1.5977, w = 2.011, b = -0.232
Epoch 70: Loss = 1.5695, w = 2.010, b = -0.215
Epoch 80: Loss = 1.5424, w = 2.010, b = -0.199
Epoch 90: Loss = 1.5163, w = 2.010, b = -0.183
Epoch 100: Loss = 1.4912, w = 2.010, b = -0.167

Final Params -> w: 2.010, b: -0.167
Target Params -> w: 2.000, b: 0.700


### with torch.no_grad():: This is mandatory when updating weights manually.

Why? If you do w = w - lr * grad without this context manager, PyTorch thinks this subtraction is part of the neural network logic and tries to build a graph for it. This leads to memory leaks and errors in the next backward pass.

In [6]:
w.item()

-0.10592065006494522

You define a class that inherits from nn.Module. This class has two requirements:

__init__: Define the components (layers) you need.

forward: Define how data flows through those components.

1. The "Input Shape" Shock
In Keras: Dense(64) (Keras figures out the input shape when you pass data). In PyTorch: nn.Linear(in_features=784, out_features=64) (You must explicitly state inputs).

Why? It forces you to know exactly what your tensor shapes are at every stage. It reduces "magic" errors later.

2. The Dynamic Forward Pass
The forward method is just Python code. You can use loops, if statements, print statements, or call other functions inside it. This is how researchers build dynamic networks (e.g., skipping layers based on conditions).

In [9]:
#MLP
import torch.nn as nn
import torch.nn.functional as F

class ResearcherNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ResearcherNet, self).__init__()
        
        # 1. Define the LAYERS (The Ingredients)
        # Note: We don't connect them here. We just list them.
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.layer2 = nn.Linear(hidden_size, hidden_size)
        self.out_layer = nn.Linear(hidden_size, output_size)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        # 2. Define the CONNECTIVITY (The Recipe)
        
        # Input -> Layer 1 -> ReLU
        x = F.relu(self.layer1(x))
        
        # Research Trick: Conditional computation
        # (Impossible in standard Keras Sequential)
        if x.mean() > 0.0: 
            x = self.dropout(x)
            x = self.layer2(x)
            x = F.relu(x)
        else:
            # Skip layer 2 if mean activation is low
            pass 
        
        # Output layer (No activation if using CrossEntropyLoss later)
        x = self.out_layer(x)
        return x

# Initialize
model = ResearcherNet(input_size=10, hidden_size=20, output_size=2)

# Inspect structure
print(model)

ResearcherNet(
  (layer1): Linear(in_features=10, out_features=20, bias=True)
  (layer2): Linear(in_features=20, out_features=20, bias=True)
  (out_layer): Linear(in_features=20, out_features=2, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


Key Note on nn.Functional vs nn.Layer:

### nn.Linear, nn.Conv2d, nn.Dropout: These are classes. They have internal state (weights) and are defined in __init__.

### F.relu, F.softmax: These are functions. They don't have weights. You can call them directly in forward.

1. class MyModel(nn.Module): (Inheritance)
What: We are creating a blueprint called MyModel.

Why (nn.Module)? We are "inheriting" from PyTorch's base class nn.Module. This gives our class superpowers. It tells PyTorch: "Hey, this isn't just a normal Python script; this contains neural network weights that you need to track, update with gradients, and save to disk."

2. def __init__(self, ...): (The Constructor)
What: This runs once when you first create the object (e.g., model = MyModel()).

Purpose: This is where you set up the inventory. You define the layers you plan to use later. You are not connecting them yet; you are just buying the parts.

Arguments: You can pass anything here. This is how you make models dynamic.

Pass dropout_rate to change regularization easily.

Pass num_layers to use a loop to create layers dynamically.

3. super(MyModel, self).__init__() (The "Magic" Line)
The Short Answer: It is mandatory boilerplate.

The "Researcher" Answer: nn.Module (the parent) has a lot of internal setup to do. It needs to create internal dictionaries to track your weights (_parameters), your sub-layers (_modules), and your buffers.

If you forget this line, when you try to assign self.layer1 = ..., PyTorch will throw an error because the internal tracking systems weren't turned on. It effectively "boots up" the PyTorch engine inside your class.

4. self.layer1 = ... (Registration)
Why self? In Python, self refers to the instance of the object.

If you write layer = nn.Linear(...) (without self), that layer is a local variable. As soon as __init__ finishes, that variable is thrown away.

If you write self.layer = nn.Linear(...), that layer sticks to the model forever.

The Magic Side Effect: Because you ran super().__init__(), whenever you assign an nn.Module (like Linear or Conv2d) to self, PyTorch automatically adds those weights to the list of things to be trained.

In [10]:
class ConfigurableNet(nn.Module):
    # We pass arguments here so we don't hardcode numbers!
    def __init__(self, input_dim, hidden_dim, use_dropout=False):
        
        # 1. Boot up the parent nn.Module logic
        super().__init__() 
        
        # 2. Define components based on arguments
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
        
        # We save this boolean to 'self' so we can check it 
        # later in the forward() method
        self.use_dropout = use_dropout 
        
        if use_dropout:
            self.dropout = nn.Dropout(p=0.5)
            
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        
        # We can use normal Python logic here!
        if self.use_dropout:
            x = self.dropout(x)
            
        x = self.fc2(x)
        return x

# NOW we create the instance. 
# __init__ runs now.
model_A = ConfigurableNet(input_dim=10, hidden_dim=50, use_dropout=True)
model_B = ConfigurableNet(input_dim=10, hidden_dim=128, use_dropout=False)

print("Model A parameters:", len(list(model_A.parameters()))) # More params (due to hidden_dim)

Model A parameters: 4
